# Machine Learning Essentials – Exercise 5

## Task 3: Automatic feature selection for LDA as regression

In [1]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

### 3.1 Implement Orthogonal Matching Pursuit (OMP)

In [2]:
def omp_regression(X, y, T):
    """
    Implements Orthogonal Matching Pursuit
    
    Input:
        X : N x D data matrix
        y : N-dimensional response vector
        T : number of non-zero coefficients desired
        
    Output:
        solutions : D x T matrix where column t is beta^(t)
                    inactive coefficients are explicitly set to zero
    """
    
    N, D = X.shape
    
    # Initialization
    A = []                     # active set A^(0)
    B = list(range(D))         # inactive set B^(0)
    r = y.copy()               # residual r^(0)
    
    solutions = np.zeros((D, T))  # final output matrix
    
    # Iteration t = 1 ... T
    for t in range(1, T+1):
        
        # Step 1: find j^(t)
        correlations = [abs(np.dot(X[:, j].T, r)) for j in B]
        j_t = B[np.argmax(correlations)]
        
        # Step 2: move j^(t) from B to A
        A.append(j_t)
        B.remove(j_t)
        
        # Step 3: form active matrix X^(t)
        X_t = X[:, A]
        
        # Step 4: solve least squares problem
        # beta^(t) = argmin (y - X^(t) beta)^T (y - X^(t) beta)
        beta_t, _, _, _ = np.linalg.lstsq(X_t, y, rcond=None)
        
        # Step 5: update residual
        r = y - X_t @ beta_t
        
        # Store full D-dimensional beta with zeros in inactive positions
        full_beta = np.zeros(D)
        for idx, col in enumerate(A):
            full_beta[col] = beta_t[idx]
        
        solutions[:, t-1] = full_beta
    
    return solutions

### 3.2 Classification with sparse LDA

In [ ]:
# Load digits dataset (scikit-learn)
digits = load_digits()

# Select only digits '3' and '9'
mask = (digits.target == 3) | (digits.target == 9)
X_full = digits.images[mask]
y_full = digits.target[mask]

# Flatten images into rows X_i
X = X_full.reshape(X_full.shape[0], -1)

# Convert labels to +1 / -1
y = np.where(y_full == 3, 1, -1)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y
)

# Optional: Standardization (we test both)
scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train)
X_test_std = scaler.transform(X_test)

# Choose T (max number of active pixels)
T = 50

# Run OMP on standardized data
solutions = omp_regression(X_train_std, y_train, T)

#### Report the error rate on the test set for t = 1...T

In [4]:
error_rates = []

for t in range(T):
    beta_t = solutions[:, t]
    y_pred = np.sign(X_test_std @ beta_t)
    error = np.mean(y_pred != y_test)
    error_rates.append(error)

# Print error rates
for t, err in enumerate(error_rates, start=1):
    print(f"t = {t:2d}, error rate = {err:.4f}")

t =  1, error rate = 0.2202
t =  2, error rate = 0.0642
t =  3, error rate = 0.0367
t =  4, error rate = 0.0367
t =  5, error rate = 0.0275
t =  6, error rate = 0.0183
t =  7, error rate = 0.0275
t =  8, error rate = 0.0183
t =  9, error rate = 0.0183
t = 10, error rate = 0.0183
t = 11, error rate = 0.0183
t = 12, error rate = 0.0183
t = 13, error rate = 0.0183
t = 14, error rate = 0.0183
t = 15, error rate = 0.0183
t = 16, error rate = 0.0183
t = 17, error rate = 0.0183
t = 18, error rate = 0.0183
t = 19, error rate = 0.0092
t = 20, error rate = 0.0092
t = 21, error rate = 0.0092
t = 22, error rate = 0.0092
t = 23, error rate = 0.0092
t = 24, error rate = 0.0092
t = 25, error rate = 0.0183
t = 26, error rate = 0.0183
t = 27, error rate = 0.0183
t = 28, error rate = 0.0092
t = 29, error rate = 0.0092
t = 30, error rate = 0.0183
t = 31, error rate = 0.0183
t = 32, error rate = 0.0183
t = 33, error rate = 0.0183
t = 34, error rate = 0.0183
t = 35, error rate = 0.0183
t = 36, error rate =

#### Visualization of pixel activation order and sign

In [ ]:
# Determine activation order
activation_order = np.zeros(X_train.shape[1], dtype=int)

for t in range(T):
    beta_t = solutions[:, t]
    active_pixels = np.where(beta_t != 0)[0]
    for p in active_pixels:
        if activation_order[p] == 0:
            activation_order[p] = t

# Determine pixel vote direction (3 or 9)
pixel_vote = np.zeros(X_train.shape[1])
pixel_vote = np.sign(solutions[:, -1])  # final solution beta^(T)

# Reshape for visualization
activation_image = activation_order.reshape(8, 8)
vote_image = pixel_vote.reshape(8, 8)

print("\nActivation order image (8x8):")
print(activation_image)

print("\nPixel vote image (8x8):")
print(vote_image)


Activation order image (8x8):
[[ 0 43 33  0  0  5 38  0]
 [30 13 14 46 45 19 42 37]
 [ 0 22  3  7 28  8 17 35]
 [ 0 27  1 10 11  1 15  0]
 [ 0 31  9 36  2  4 25  0]
 [ 0 32  6 41 18 20 16  0]
 [ 0 47 44 21 29 12 39 49]
 [ 0  0 34 24 48 26 23 40]]

Pixel vote image (8x8):
[[ 0.  1. -1.  0.  0.  1.  1.  0.]
 [-1. -1. -1.  1. -1.  1. -1. -1.]
 [ 0.  1. -1. -1. -1. -1.  1.  1.]
 [ 0. -1. -1. -1.  1. -1. -1.  0.]
 [ 0. -1. -1. -1.  1.  1.  1.  0.]
 [ 0.  1.  1.  1.  1.  1.  1.  0.]
 [ 0. -1.  1. -1.  1. -1.  1.  1.]
 [ 0.  0.  1. -1. -1.  1. -1.  1.]]
